# Lab 4 — Why It Matters for Agents: Do You Even Need a Model?

**AIEWF 2026 Workshop — Vector → Hybrid → Do You Even Need a Model?**

This notebook wires the Lab 3 hybrid retriever to an LLM synthesis call.

The thesis: **retrieval quality — not model quality — determines answer quality.**

---

## Requirements
```
# requirements.txt
elasticsearch>=8.17.0
requests
```

Environment variables needed (pre-configured in Instruqt sandbox):
- `ES_ENDPOINT` — your Elastic Serverless project URL
- `ES_API_KEY` — your Elastic API key

**No separate LLM key needed.** LLM synthesis uses the Elastic Inference API — your
ES credentials cover both search and LLM calls.


In [ ]:
# Cell 1 — Install and Import

# Uncomment to install if needed:
# !pip install elasticsearch requests

import os
import json
import requests
from elasticsearch import Elasticsearch

print('Imports OK')


In [ ]:
# Cell 2 — Connect to Elasticsearch
# Uses the same Serverless project as Labs 1-3.

ES_ENDPOINT = os.environ.get('ES_ENDPOINT')
ES_API_KEY = os.environ.get('ES_API_KEY')

if not ES_ENDPOINT or not ES_API_KEY:
    raise ValueError(
        'Set ES_ENDPOINT and ES_API_KEY environment variables.\n'
        'In Instruqt: these are pre-configured in the sandbox environment.\n'
        'Running locally: export ES_ENDPOINT=https://... ES_API_KEY=...'
    )

es = Elasticsearch(ES_ENDPOINT, api_key=ES_API_KEY, request_timeout=60)

info = es.info()
print(f'Connected to Elasticsearch {info["version"]["number"]}')

# Verify corpus is indexed
count = es.count(index='aiewf-workshop-docs')['count']
print(f'Index has {count} documents')

In [ ]:
# Cell 3 — The Lab 3 Hybrid Retriever as a Python Function
#
# Same RRF retriever DSL from Lab 3 Dev Console, now as Python.
# ES client's search() takes the same body dict you'd paste into Dev Console.

INDEX_NAME = 'aiewf-workshop-docs'

def hybrid_search(query: str, size: int = 5) -> list[dict]:
    """
    Run a hybrid RRF search combining BM25 + semantic.
    
    Uses the same RRF retriever built in Lab 3.
    Returns a list of dicts: {id, title, url, body (truncated), score}
    """
    response = es.search(
        index=INDEX_NAME,
        body={
            "retriever": {
                "rrf": {
                    "retrievers": [
                        {
                            "standard": {
                                "query": {
                                    "multi_match": {
                                        "query": query,
                                        "fields": ["title^3", "body"],
                                        "type": "best_fields"
                                    }
                                }
                            }
                        },
                        {
                            "standard": {
                                "query": {
                                    "semantic": {
                                        "field": "body_semantic",
                                        "query": query
                                    }
                                }
                            }
                        }
                    ],
                    "rank_constant": 60,
                    "rank_window_size": 100
                }
            },
            "size": size,
            "_source": ["id", "title", "url", "body"]
        }
    )
    
    results = []
    for hit in response['hits']['hits']:
        src = hit['_source']
        results.append({
            'id': src['id'],
            'title': src['title'],
            'url': src.get('url', ''),
            'body': src['body'][:800],  # truncate for context window management
            'score': hit['_score']
        })
    return results


# Quick test
results = hybrid_search("notify me when something goes wrong", size=3)
print(f'Hybrid search returned {len(results)} results:')
for r in results:
    print(f'  [{r["id"]}] {r["title"]} (score: {r["score"]:.4f})')

In [ ]:
# Cell 4 — LLM Synthesis Function
#
# Uses the Elastic Inference API — no separate LLM key needed.
# Your ES_API_KEY covers both search and LLM inference.
# We use Claude Haiku 4.5 (fast, cheap) — even it synthesizes a clear answer
# when context is correct (Cell 5), and fails when context is wrong (Cell 6).
#
# NOTE: we use the `completion` task type (not `chat_completion`). On Serverless the
# chat_completion task type only supports the streaming (_stream) API; the `completion`
# task type returns a single JSON response, which is simpler for a notebook.

INFERENCE_ID = '.anthropic-claude-4.5-haiku-completion'

SYSTEM_PROMPT = (
    'You are a helpful Elasticsearch documentation assistant. '
    'Answer the question using ONLY the provided documentation context. '
    'If the context does not contain enough information, say so clearly.'
)


def synthesize(context_docs: list[dict], question: str, inference_id: str = INFERENCE_ID) -> str:
    context_parts = []
    for i, doc in enumerate(context_docs, 1):
        context_parts.append(f"[Document {i}: {doc['title']}]\n{doc['body']}\n")
    context_text = '\n---\n'.join(context_parts)

    # The `completion` task type takes a single `input` string, so we fold the
    # system instructions, context, and question into one prompt.
    prompt = f"{SYSTEM_PROMPT}\n\nContext:\n{context_text}\n\nQuestion: {question}"

    resp = requests.post(
        f"{ES_ENDPOINT}/_inference/completion/{inference_id}",
        headers={
            'Authorization': f'ApiKey {ES_API_KEY}',
            'Content-Type': 'application/json',
        },
        json={'input': prompt},
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()['completion'][0]['result']


print(f'synthesize() defined → Elastic Inference API ({INFERENCE_ID})')
print('No ANTHROPIC_API_KEY needed — ES credentials cover LLM inference too.')

## Cell 5 — Good Context Demo (PRE-BAKED)

This cell uses a **hardcoded question** and **pre-retrieved good context** — the kind of docs the hybrid retriever surfaces for "how do I get notified when something goes wrong" (the Watcher alerting page — doc-049 — plus the cluster-health and cat-API docs that an alert would actually watch).

**Run it as-is. Do not modify the context.**

The context is pre-baked so this demo is deterministic — the answer below was generated by the same model (Haiku 4.5) on exactly this context before the workshop.

### Why this works:
The hybrid retriever delivers docs that are genuinely *about* alerting. The Watcher page explains triggers, conditions, and actions; the cluster-health page describes the red/yellow signal a watch would fire on. The model synthesizes them into a concrete, correct answer. Even cheap Haiku does this well because the context is good.

In [ ]:
# Cell 5 — GOOD CONTEXT DEMO (pre-baked)
# DO NOT MODIFY — context is hardcoded for reproducibility

DEMO_QUESTION = "How do I get notified when something goes wrong in my cluster?"

# Pre-retrieved good context from the hybrid retriever
# (Real excerpts from the workshop corpus — verified to produce a good answer)
GOOD_CONTEXT = [
    {
        "id": "doc-049",
        "title": "Elasticsearch Watcher alerting",
        "url": "https://www.elastic.co/docs/explore-analyze/alerting/watcher",
        "body": (
            "Watcher is Elasticsearch's built-in alerting and notification system. Watches monitor "
            "conditions in your data and trigger actions when thresholds are met. A watch consists of "
            "a trigger (when it runs, typically a schedule), an input (what data to load, such as a "
            "search or an HTTP request to /_cluster/health), a condition (logic that decides whether "
            "to act, for example cluster status equals red), and actions (what to do when the condition "
            "is true: email, webhook, Slack, or index). Kibana Alerting (Rules and Connectors) is the "
            "modern alternative with a UI and more rule types; Watcher remains available for API-driven "
            "alert management."
        )
    },
    {
        "id": "doc-021",
        "title": "Red or yellow cluster health status troubleshooting",
        "url": "https://www.elastic.co/docs/troubleshoot/elasticsearch/red-yellow-cluster-status",
        "body": (
            "A red or yellow cluster health status indicates one or more shards are not assigned to a "
            "node. Red means unassigned primary shards (searches and indexing may fail); yellow means "
            "some replica shards are unassigned. Check GET /_cluster/health — a healthy cluster shows "
            "status green and unassigned_shards 0. A change to red or yellow is exactly the kind of "
            "condition an alert should watch for."
        )
    },
    {
        "id": "doc-043",
        "title": "Elasticsearch cat APIs overview",
        "url": "https://www.elastic.co/docs/reference/elasticsearch/rest-apis/compact-aliases",
        "body": (
            "The cat APIs return compact, human-readable operational views. GET /_cat/health shows "
            "cluster status; GET /_cat/nodes shows per-node CPU, heap, and load; GET /_cat/shards shows "
            "shard allocation and unassigned reasons. These endpoints are commonly polled by monitoring "
            "tools or used as a watch input to detect problems early."
        )
    }
]

print(f'Question: {DEMO_QUESTION}')
print(f'Context: {len(GOOD_CONTEXT)} documents (pre-baked good context)')
print('\n--- Answer (pre-generated by Haiku 4.5 on this exact context) ---\n')

GOOD_ANSWER = "Based on the provided documentation, here are the main ways to get notified when something goes wrong in your cluster:\n\n## Using Watcher (API-driven alerting)\n\nElasticsearch's built-in **Watcher** alerting system is designed for this purpose. A watch monitors conditions in your data and triggers actions when thresholds are met. A typical watch consists of:\n\n1. **Trigger** — defines when it runs (typically on a schedule)\n2. **Input** — loads data to check, such as a search or a request to `/_cluster/health`\n3. **Condition** — logic that decides whether to act (for example, cluster status equals red)\n4. **Actions** — what to do when the condition is true, including:\n   - Email\n   - Webhook\n   - Slack\n   - Index a document\n\n## Example Use Case\n\nFor cluster health monitoring specifically, you could create a watch that:\n- Uses `GET /_cluster/health` as input to check cluster status\n- Monitors for red or yellow status (which indicate unassigned shards)\n- Sends notifications via email, Slack, or webhook when problems are detected\n\n## Modern Alternative\n\nThe documentation also mentions that **Kibana Alerting (Rules and Connectors)** is a modern alternative with a UI and more rule types, though Watcher remains available for API-driven alert management."
print(GOOD_ANSWER)

## Cell 6 — Bad Context Demo (PRE-BAKED)

**Same question. Same model. Same prompt. Different context.**

We replaced the retrieved docs with irrelevant documents about ILM lifecycle policies and snapshot configuration.

### The model didn't get dumber. The retrieval got worse.

This is the core thesis of the workshop. The LLM is a synthesis engine — it works with what you give it. Bad retrieval → bad context → bad answer. The model is not the failure point.

In [ ]:
# Cell 6 — BAD CONTEXT DEMO (pre-baked)
# Same question, same model, same prompt — deliberately wrong context
# DO NOT MODIFY — context is hardcoded to produce a bad answer reproducibly

# Same question as Cell 5
# DEMO_QUESTION = "How do I get notified when something goes wrong in my cluster?"

# Deliberately wrong context: ILM, snapshots, ingest pipelines — nothing about alerting
BAD_CONTEXT = [
    {
        "id": "doc-017",
        "title": "Index lifecycle management overview",
        "url": "https://www.elastic.co/docs/manage-data/lifecycle/index-lifecycle-management",
        "body": (
            "Index Lifecycle Management (ILM) automates index management through hot, warm, cold, "
            "frozen, and delete phases. Define a policy and apply it via an index template; use the "
            "rollover action to create new indices when the current one reaches a size or age threshold."
        )
    },
    {
        "id": "doc-037",
        "title": "Snapshot and restore overview",
        "url": "https://www.elastic.co/docs/deploy-manage/tools/snapshot-and-restore",
        "body": (
            "Snapshots back up cluster data to a repository (S3, GCS, Azure, or shared filesystem). "
            "Create a repository, take a snapshot, and restore from it. Snapshot lifecycle management "
            "(SLM) automates scheduled snapshots."
        )
    },
    {
        "id": "doc-019",
        "title": "Ingest pipelines in Elasticsearch",
        "url": "https://www.elastic.co/docs/manage-data/ingest/transform-enrich/ingest-pipelines",
        "body": (
            "Ingest pipelines transform documents before indexing using processors such as set, grok, "
            "date, rename, and script. Create a pipeline with PUT /_ingest/pipeline and apply it via "
            "the index default_pipeline setting or per request."
        )
    }
]

print(f'Question: {DEMO_QUESTION}')
print(f'Context: {len(BAD_CONTEXT)} documents (WRONG context — ILM, snapshots, pipelines)')
print('\n--- Answer (pre-generated by Haiku 4.5 on this exact context) ---\n')

BAD_ANSWER = "I don't have enough information in the provided documentation to answer your question about cluster notifications or alerts.\n\nThe documentation context covers:\n- Index Lifecycle Management (ILM)\n- Snapshots and restore\n- Ingest pipelines\n\nNone of these sections address cluster monitoring, alerting, or notification mechanisms. To answer your question properly, I would need documentation covering topics like Elasticsearch alerting, monitoring, or watcher functionality.\n\nI recommend checking the Elasticsearch documentation for sections on \"Alerting,\" \"Watcher,\" \"Stack Monitoring,\" or \"Notifications\" for this information."
print(BAD_ANSWER)

print('\n' + '='*60)
print('THE LESSON: The model did not get dumber. The retrieval got worse.')
print('Retrieval quality determines answer quality.')

In [ ]:
# Cell 7 — Full Pipeline End-to-End  [INSTRUCTOR DEMO — runs on screen]
#
# hybrid_search() + synthesize() both use ES credentials — nothing else needed.

def ask(question: str, inference_id: str = INFERENCE_ID) -> str:
    """Full RAG pipeline: hybrid retrieve → LLM synthesize via Elastic Inference API."""
    print(f'Retrieving context for: "{question}"')
    docs = hybrid_search(question, size=5)
    print(f'Retrieved {len(docs)} docs:')
    for d in docs:
        print(f'  [{d["id"]}] {d["title"]}')
    print('\nSynthesizing via Elastic Inference API...')
    return synthesize(docs, question, inference_id=inference_id)


answer = ask("notify me when something goes wrong")
print('\n--- Answer ---')
print(answer)

In [ ]:
# Cell 7b — Try your own questions  [TAKE-HOME]
# Requires Cell 7 to have run first (ask() must be defined).

YOUR_QUESTION = "how do I set up TLS for my cluster?"  # Change this

answer = ask(YOUR_QUESTION)
print('\n--- Answer ---')
print(answer)


## Cell 8 — Multi-Hop Agent Loop

Single-shot RAG answers one question from one retrieval. Some questions need **two lookups**: find one fact, then use it to decide what to search for next — e.g. *"my cluster is yellow with unassigned shards — what causes that, and how do I use the allocation explain API to fix it?"* Hop 1 explains the symptom; the agent then searches again for the allocation-explain mechanics.

The cell below is a minimal multi-hop agent. Two things make the difference between it actually taking a second hop and silently stopping at one — the naive version gets both wrong:

1. **The decision prompt must *invite* a follow-up.** Each turn the model replies `ANSWER: …` (context is enough) or `LOOKUP: <query>` (a sub-fact is missing). Telling a model "answer if you can" biases it to one-shot everything; we explicitly tell it to `LOOKUP` when a two-part question is only half-covered.
2. **Parse the decision robustly.** Models dress up output (`# ANSWER:`, `**LOOKUP:**`); a bare `startswith("ANSWER:")` misses those. We strip markdown and match the first `ANSWER:`/`LOOKUP:` token case-insensitively.

This is the hand-rolled version — you own the loop, the prompt, and the parser. **Part 3** builds the same behavior in Elastic Agent Builder, where the platform runs the loop for you.

In [ ]:
# Cell 8 — Multi-Hop Agent Loop
# The agent decides ANSWER vs LOOKUP each turn; on a two-part question it takes a second hop.
# Uses the same `completion` task type as synthesize() — no extra keys.

import re

DECISION_PROMPT = (
    "You are a multi-hop retrieval agent. You are given a user question and the documents "
    "retrieved for it so far. Decide between two actions and respond in a STRICT format:\n"
    "- If the retrieved context fully answers the question, reply: ANSWER: <the answer>\n"
    "- If a key sub-fact is missing (a setting, root cause, threshold, or related subsystem the "
    "context points to but does not fully explain), reply: LOOKUP: <a short search query>\n"
    "Rules:\n"
    "- Your reply MUST begin with the literal token ANSWER: or LOOKUP: — no markdown, no '#'.\n"
    "- A LOOKUP must be a SHORT search query on one line (a few keywords), not a sentence.\n"
    "- Prefer LOOKUP when the question has two parts and the current context covers only one.\n"
    "- Use at most one LOOKUP, then ANSWER from the combined context."
)

def _completion(prompt: str) -> str:
    """Call the Elastic Inference `completion` task type; return the text result."""
    resp = requests.post(
        f"{ES_ENDPOINT}/_inference/completion/{INFERENCE_ID}",
        headers={'Authorization': f'ApiKey {ES_API_KEY}', 'Content-Type': 'application/json'},
        json={'input': prompt},
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()['completion'][0]['result']

def parse_action(text: str):
    """Extract (kind, payload) robustly: strip markdown, match first ANSWER:/LOOKUP: token."""
    cleaned = text.lstrip("#*>- \n\t")
    m = re.search(r"(ANSWER|LOOKUP)\s*:", cleaned, re.IGNORECASE)
    if not m:
        return ("ANSWER", text.strip())
    payload = cleaned[m.end():].strip()
    if m.group(1).upper() == "LOOKUP" and payload:
        payload = payload.splitlines()[0].strip()   # keep only the query line
    return (m.group(1).upper(), payload)

def _context_block(docs):
    return '\n\n---\n\n'.join(f"[{d['title']}]\n{d['body']}" for d in docs)

def multi_hop_agent(initial_question: str, max_hops: int = 2) -> str:
    """Minimal multi-hop RAG agent: retrieve → decide (ANSWER/LOOKUP) → maybe retrieve again."""
    print(f'Question: "{initial_question}"\n')
    question = initial_question
    history = ""
    for hop in range(1, max_hops + 1):
        print(f'--- Hop {hop}: retrieving for "{question}" ---')
        docs = hybrid_search(question, size=3)
        for d in docs:
            print(f'  [{d["id"]}] {d["title"]}')

        decision_input = (
            f"{DECISION_PROMPT}\n\n"
            + (f"Earlier findings:\n{history}\n\n" if history else "")
            + f"Question: {initial_question}\n\nRetrieved context:\n{_context_block(docs)}"
        )
        kind, payload = parse_action(_completion(decision_input))
        history += f"\n\n[Hop {hop} context]\n{_context_block(docs)}"

        if kind == "ANSWER":
            print(f'\nAgent decided: ANSWER (after {hop} hop{"s" if hop > 1 else ""})\n')
            return payload
        print(f'\nAgent decided: LOOKUP → "{payload}"\n')
        question = payload

    # Hit the hop limit still wanting more — synthesize from everything gathered
    print(f'Reached max_hops={max_hops}; synthesizing from combined context...\n')
    return _completion(f"{SYSTEM_PROMPT}\n\nContext:\n{history}\n\nQuestion: {initial_question}")


result = multi_hop_agent(
    "My cluster health is yellow with unassigned shards — what causes that, "
    "and how do I use the allocation explain API to fix it?"
)
print('=== Final Answer ===')
print(result)

---

# Part 3 — The same agent, in Elastic Agent Builder

You just hand-rolled the loop, the decision prompt, and the parser. That's how you *understand* an agent — not how you *ship* one. **Elastic Agent Builder** runs that loop for you: you supply a **tool** (the retrieval step) and an **agent** (a system prompt + its tools), and the platform handles reason → call tool → read → call again → answer, including multi-hop.

We register your Lab 3 hybrid retriever as the tool. In ES|QL the RRF fusion is one statement — `FORK` runs the BM25 and semantic arms, `FUSE` applies RRF — returning the same ranking as the `rrf` retriever you've used all workshop:

```esql
FROM aiewf-workshop-docs METADATA _score, _id, _index
| FORK ( WHERE match(body, ?query)          | SORT _score DESC | LIMIT 50 )
       ( WHERE match(body_semantic, ?query) | SORT _score DESC | LIMIT 50 )
| FUSE | SORT _score DESC | KEEP id, title, url, body, _score | LIMIT 5
```

Agent Builder is a **Kibana** API, so the next cell needs `KIBANA_URL` (the `.kb.` endpoint) — the same `ES_API_KEY` authenticates it. In the sandbox the tool + agent are pre-created at startup; the cell is idempotent, so re-running it is safe.

In [ ]:
# Part 3 — Create the Agent Builder tool + agent, then run it (idempotent)
KIBANA_URL = os.environ.get('KIBANA_URL')
if not KIBANA_URL:
    raise ValueError(
        'Set KIBANA_URL — the Kibana endpoint (.kb.), not Elasticsearch (.es.).\n'
        '  In Instruqt: pre-configured. Repo: export KIBANA_URL=https://...kb...:443'
    )

AB_TOOL_ID, AB_AGENT_ID = 'search-workshop-docs-hybrid', 'workshop-docs-agent'

HYBRID_ESQL = (
    'FROM aiewf-workshop-docs METADATA _score, _id, _index\n'
    '| FORK ( WHERE match(body, ?query) | SORT _score DESC | LIMIT 50 )\n'
    '       ( WHERE match(body_semantic, ?query) | SORT _score DESC | LIMIT 50 )\n'
    '| FUSE\n| SORT _score DESC\n| KEEP id, title, url, body, _score\n| LIMIT 5'
)

AGENT_INSTRUCTIONS = """You are an Elasticsearch documentation assistant built on hybrid retrieval.

## Your tool
- **search-workshop-docs-hybrid**: hybrid (BM25 + semantic, RRF-fused) search over the Elasticsearch docs corpus. Returns the top 5 docs with id, title, url, and body.

## How you work (multi-hop)
1. Call search-workshop-docs-hybrid with the user's question to get an initial set of docs.
2. If they fully answer the question, write the answer.
3. If a key sub-fact is missing (a setting, root cause, or related subsystem the first results point to but don't fully cover), run a SECOND search with a refined query, then answer from the combined results. Prefer one focused follow-up.
4. Ground every claim in retrieved docs. Cite sources as [title]. If the docs don't answer it, say so.

## Style
Concise and technical. Name the specific settings, codes, or commands the docs mention."""

def _ab(method, path, body=None):
    r = requests.request(method, f"{KIBANA_URL}{path}",
        headers={'Authorization': f'ApiKey {ES_API_KEY}', 'kbn-xsrf': 'true',
                 'Content-Type': 'application/json'}, json=body, timeout=60)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text

# Idempotent reset (agent references tool -> delete agent first)
_ab('DELETE', f'/api/agent_builder/agents/{AB_AGENT_ID}')
_ab('DELETE', f'/api/agent_builder/tools/{AB_TOOL_ID}')

s, _ = _ab('POST', '/api/agent_builder/tools', {
    'id': AB_TOOL_ID, 'type': 'esql',
    'description': ('Hybrid (BM25 + semantic) search over the Elasticsearch docs corpus, fused with '
                    'RRF. Use for ANY question about Elasticsearch. Returns top 5 docs (id, title, '
                    'url, body). Call again with a refined query if the first results fall short.'),
    'tags': ['workshop', 'search', 'hybrid', 'rag'],
    'configuration': {'query': HYBRID_ESQL,
                      'params': {'query': {'type': 'string',
                                           'description': 'The search query or keywords.'}}},
})
print(f"{'OK' if s == 200 else 'FAIL'} tool  '{AB_TOOL_ID}' (HTTP {s})")

s, _ = _ab('POST', '/api/agent_builder/agents', {
    'id': AB_AGENT_ID, 'name': 'Workshop Docs Agent',
    'description': 'Multi-hop RAG agent over the workshop docs corpus, powered by hybrid retrieval.',
    'labels': ['workshop'],
    'configuration': {'instructions': AGENT_INSTRUCTIONS, 'tools': [{'tool_ids': [AB_TOOL_ID]}]},
})
print(f"{'OK' if s == 200 else 'FAIL'} agent '{AB_AGENT_ID}' (HTTP {s})")

# Run it on a two-part question and show the multi-hop tool calls
print('\nAsking the agent a two-part question (runs server-side; ~15-25s)...\n')
s, result = _ab('POST', '/api/agent_builder/converse', {
    'agent_id': AB_AGENT_ID,
    'input': ('My Elasticsearch container keeps dying with exit code 137. Why does that happen, '
              'and what JVM and memory settings should I change to prevent it?'),
})
if s != 200:
    raise RuntimeError(f'converse failed (HTTP {s}): {result}')

tool_calls = 0
for step in result.get('steps', []):
    if step.get('type') == 'tool_call':
        tool_calls += 1
        print(f"  hop {tool_calls}: {step.get('tool_id')}  query={step.get('params', {}).get('query')!r}")

resp = result.get('response', '')
answer = resp if isinstance(resp, str) else (resp.get('message') if isinstance(resp, dict)
         else ''.join(b.get('text', '') for b in resp if isinstance(b, dict)))
print(f"\nTool calls (retrieval hops): {tool_calls}")
print('\n=== AGENT ANSWER ===')
print(answer)
print(f"\nNow open Agent Builder in Kibana and chat with 'Workshop Docs Agent':\n  {KIBANA_URL}/app/agent_builder")


## Now run it yourself in Kibana

Switch to the **Agent Builder** tab (or `${KIBANA_URL}/app/agent_builder`), pick **Workshop Docs Agent**, and ask a two-part question:

- *"My cluster is yellow with unassigned shards — what causes it and how do I use the allocation explain API to fix it?"*
- *"Container exits with code 137 — why, and which JVM heap settings prevent it?"*

Each tool call in the conversation is a hybrid-retrieval hop — on a two-part question the agent searches, reads, and searches **again** before answering. Click a tool call to inspect the ES|QL it ran and the docs it got back: that's your Lab 3 retriever, inside the agent.

---

**The same retriever powered all three levels** — single-shot RAG, the hand-rolled loop, and this Agent Builder agent. The agent framework is the easy part to swap; retrieval quality — what you measured in Labs 2 and 3 — is what determines whether any of it works.